# 🧬 AMR-Predictor-ML: Predicting Meropenem Resistance in *Klebsiella pneumoniae*
## Phase 1: Data Exploration, Cleaning, and Dataset Preparation

### 📋 Project Overview
Antimicrobial Resistance (AMR) is a critical global health threat. This project leverages **Machine Learning** to predict whether a specific strain of *Klebsiella pneumoniae* is **Resistant** or **Susceptible** to **Meropenem** (a last-resort Carbapenem antibiotic) directly from its genomic data.

---

### 📊 Dataset Summary (Exploratory Data Analysis - EDA)
Based on our initial data profile from the PATRIC/BV-BRC database:
- **Organism:** *Klebsiella pneumoniae*
- **Antibiotic Target:** Meropenem (Carbapenem family)
- **Total Samples:** 3,064 bacterial strains
- **Class Distribution:**
  - 🟢 **Susceptible (S):** 1,876 strains (61.2%)
  - 🔴 **Resistant (R):** 1,188 strains (38.8%)
  
*Note: The dataset is well-balanced, minimizing the risk of severe class imbalance during model training.*

---

### 🎯 Notebook Objectives
1. Load and merge the resistant and susceptible metadata.
2. Clean out irrelevant clinical/computational columns to reduce noise.
3. Standardize the `Genome ID` column to act as the primary key for downloading genomic sequences (FASTA files).
4. Export a cleaned master metadata file for the feature engineering phase.

---

In [1]:
import pandas as pd
import os

# 1. Define relative directory paths
RAW_DIR = "../data/raw"
PROCESSED_DIR = "../data/processed"

# 2. Read the raw metadata files
print("⏳ Loading raw metadata files...")
df_res = pd.read_csv(os.path.join(RAW_DIR, "resistant_metadata.csv"))
df_sus = pd.read_csv(os.path.join(RAW_DIR, "susceptible_metadata.csv"))

# 3. Merge datasets into a single Master DataFrame (Concatenation)
df_master = pd.concat([df_res, df_sus], ignore_index=True)
print(f"✅ Data merged successfully. Total rows: {df_master.shape[0]}")

# 4. Filter columns to keep only relevant features
columns_to_keep = ['Genome ID', 'Genome Name', 'Antibiotic', 'Resistant Phenotype']
df_clean = df_master[columns_to_keep].copy()

# 5. Perform Binary Encoding on target labels for Machine Learning
# Resistant -> 1  |  Susceptible -> 0
df_clean['AMR_Label'] = df_clean['Resistant Phenotype'].map({'Resistant': 1, 'Susceptible': 0})

# 6. Save the cleaned dataset to the processed directory
os.makedirs(PROCESSED_DIR, exist_ok=True) # Create directory if it does not exist
output_path = os.path.join(PROCESSED_DIR, "cleaned_meropenem_metadata.csv")
df_clean.to_csv(output_path, index=False)

print(f"💾 Cleaned data saved to: {output_path}")

# 7. Validate the final class distribution
print("\n📊 Final Class Distribution:")
print(df_clean['AMR_Label'].value_counts())

⏳ Loading raw metadata files...
✅ Data merged successfully. Total rows: 3064
💾 Cleaned data saved to: ../data/processed/cleaned_meropenem_metadata.csv

📊 Final Class Distribution:
AMR_Label
0    1876
1    1188
Name: count, dtype: int64


## 🧬 Subsection: Automated Genomic Data Retrieval (FASTA Download)

### 🎯 Objective
Download the complete genomic sequences (FASTA files) for the 3,064 *Klebsiella pneumoniae* strains using the BV-BRC (PATRIC) REST API. 

### ⚠️ Execution Note
Since downloading 3,000+ bacterial genomes involves fetching ~15 GB of data, this script contains built-in error handling, directory management, and an option to test a small batch before running the full download pipeline.

In [4]:
import os
import pandas as pd
import requests
from tqdm import tqdm

# 1. Define paths
PROCESSED_DIR = "../data/processed"
FASTA_DIR = "../data/raw/fasta"
os.makedirs(FASTA_DIR, exist_ok=True)

# 2. Load our cleaned metadata
metadata_path = os.path.join(PROCESSED_DIR, "cleaned_meropenem_metadata.csv")
df = pd.read_csv(metadata_path)

def download_fasta(genome_id, output_folder):
    """
    Downloads the complete FASTA genome sequence using the official public BV-BRC API.
    Uses the main domain (www.bv-brc.org) to guarantee stable DNS resolution.
    """
    file_path = os.path.join(output_folder, f"{genome_id}.fasta")
    
    # Skip if file already exists (Resume-friendly)
    if os.path.exists(file_path):
        return "Exists"
        
    # Construct the exact RQL query format required by BV-BRC: eq(genome_id, VALUE)
    url = f"https://www.bv-brc.org/api/genome_sequence/?eq(genome_id,{genome_id})&http_accept=application/dna+fasta"
    
    # Standard headers to enforce FASTA formatting response
    headers = {
        "Accept": "application/dna+fasta"
    }
    
    try:
        response = requests.get(url, headers=headers, timeout=30)
        # Check if response is successful and contains data (not empty)
        if response.status_code == 200 and len(response.text.strip()) > 0:
            with open(file_path, "w") as f:
                f.write(response.text)
            return "Success"
        else:
            return f"Failed (Status Code: {response.status_code})"
    except Exception as e:
        return f"Error: {str(e)}"

# 3. Run the Test Batch on the first 5 genomes
print("🚀 Starting a Test Batch download with the official public BV-BRC endpoint...")
test_subset = df.head(5)

for idx, row in test_subset.iterrows():
    gid = row['Genome ID']
    print(f"⏳ Downloading genome {gid} ({idx+1}/5)...")
    result = download_fasta(gid, FASTA_DIR)
    print(f"📊 Result: {result}")

print(f"\n📂 Task complete! Please check the folder '{FASTA_DIR}' for downloaded files.")

🚀 Starting a Test Batch download with the official public BV-BRC endpoint...
⏳ Downloading genome 72407.572 (1/5)...
📊 Result: Success
⏳ Downloading genome 573.24455 (2/5)...
📊 Result: Success
⏳ Downloading genome 573.14181 (3/5)...
📊 Result: Success
⏳ Downloading genome 573.67259 (4/5)...
📊 Result: Success
⏳ Downloading genome 573.14163 (5/5)...
📊 Result: Success

📂 Task complete! Please check the folder '../data/raw/fasta' for downloaded files.


### 🚀 Production Run: Full Genomic Dataset Download

#### 🎯 Objective
Execute the full genomic data retrieval pipeline to download the FASTA sequences for all 3,064 *Klebsiella pneumoniae* strains. 

#### ⚙️ Pipeline Features
- **Smart Resume (Idempotency):** The script automatically detects and skips files that are already downloaded on the disk (`Exists`). If the network disconnects, you can safely re-run the cell without starting from scratch.
- **Real-Time Monitoring:** Integrated with `tqdm` to provide a dynamic progress bar, ETA, and download speed.

*⏳ **Note:** Downloading ~3,000 bacterial genomes is a data-intensive process (fetching several gigabytes). Please leave the environment running until the progress bar reaches 100%.*

In [ ]:
import os
import pandas as pd
import requests
from tqdm import tqdm

# 1. Paths configuration
PROCESSED_DIR = "../data/processed"
FASTA_DIR = "../data/raw/fasta"
os.makedirs(FASTA_DIR, exist_ok=True)

# 2. Load the master metadata tracking sheet
metadata_path = os.path.join(PROCESSED_DIR, "cleaned_meropenem_metadata.csv")
df = pd.read_csv(metadata_path)

def download_fasta(genome_id, output_folder):
    """
    Downloads the complete FASTA genome sequence using the official BV-BRC API.
    Skips the download if the file already exists locally.
    """
    file_path = os.path.join(output_folder, f"{genome_id}.fasta")
    
    # Smart Check: Skip if file already exists (Resume-friendly framework)
    if os.path.exists(file_path):
        return "Exists"
        
    # Construct standard RQL query URL structure
    url = f"https://www.bv-brc.org/api/genome_sequence/?eq(genome_id,{genome_id})&http_accept=application/dna+fasta"
    headers = {"Accept": "application/dna+fasta"}
    
    try:
        response = requests.get(url, headers=headers, timeout=30)
        if response.status_code == 200 and len(response.text.strip()) > 0:
            with open(file_path, "w") as f:
                f.write(response.text)
            return "Success"
        else:
            return f"Failed (Status Code: {response.status_code})"
    except Exception as e:
        return f"Error: {str(e)}"

# 3. Trigger full pipeline processing for all 3,064 target sequences
print(f"🚀 Initializing production download for all {len(df)} genomic sequences...")

metrics = {"Success": 0, "Exists": 0, "Failed": 0}

# Execution loop with active progress bar feedback loop
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Fetching Bacterial Genomes"):
    gid = row['Genome ID']
    status = download_fasta(gid, FASTA_DIR)
    
    if status == "Success":
        metrics["Success"] += 1
    elif status == "Exists":
        metrics["Exists"] += 1
    else:
        metrics["Failed"] += 1

# 4. Process execution summary metadata output
print("\n📊 --- Pipeline Execution Run Summary ---")
print(f"✅ Newly Downloaded Sequences : {metrics['Success']}")
print(f"🔄 Skipped (Verified On Disk)  : {metrics['Exists']}")
print(f"❌ Connection/API Failures      : {metrics['Failed']}")

🚀 Initializing production download for all 3064 genomic sequences...


Fetching Bacterial Genomes:  65%|████████████████████████████████▌                 | 1998/3064 [00:56<00:18, 57.33it/s]